In [ ]:
''' Imports '''

import pandas as pd
import polars as pl
import numpy as np
import math

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

from scipy.stats.mstats import trimmed_var
from scipy.stats import percentileofscore

from sklearn.preprocessing import StandardScaler, Normalizer
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans, SpectralClustering, DBSCAN, HDBSCAN, OPTICS
from sklearn.metrics import silhouette_score

from scripts.prep_data import load_stats_team_tendencies_offense

# Get Data

In [ ]:
offense_tendencies = load_stats_team_tendencies_offense()

print(offense_tendencies.head().to_string())

In [ ]:
''' Features '''


FEATURES = [
    'Plays / Game', 'Drives / Game', 'Scrambles / Game', 
    '% Plays 11 Personnel', '% Under Center', 'Shotgun % Pass', 'Under Center % Pass',
    'ADOT', 'Avg Time to Throw', '% Passes Behind LOS', '% Passes Deep', 'MaxTargetShare',
    '% Rush Outside', 'MaxRushAttemptsShare',
]

# Preprocessing & Transformation

In [ ]:
''' Transform and Scale '''

# Log transform data
# transformed_data = pd.DataFrame(np.log(offense_tendencies[OFFENSE_FEATURES]), columns=OFFENSE_FEATURES).replace(math.inf, 0).replace(-(math.inf), 0)

# Scale data
scaler = StandardScaler()
scaled_data = scaler.fit_transform(offense_tendencies[FEATURES])
scaled_data_df = pd.DataFrame(scaled_data, columns=FEATURES)

print(scaled_data_df.shape)
print(scaled_data_df.head().to_string())

# Scale data
norm = Normalizer()
norm_data = norm.fit_transform(offense_tendencies[FEATURES])
norm_data_df = pd.DataFrame(norm_data, columns=FEATURES)

print(norm_data_df.shape)
print(norm_data_df.head().to_string())

In [ ]:
''' Input '''

cluster_input = norm_data_df.copy()
cluster_n_features = len(cluster_input.columns)

print(f'{cluster_n_features = }')
print(cluster_input.head().to_string())

In [ ]:
''' t-SNE '''

# Parameters
PERPLEXITY = 30
TSNE_N_COMPONENTS = 3
TSNE_N_COMPONENT_NAMES = [f'TSNE Component {n+1}' for n in range(TSNE_N_COMPONENTS)]

# Model
tsne_model = TSNE(n_components=TSNE_N_COMPONENTS, perplexity=PERPLEXITY, random_state=42)

# Fit
y = tsne_model.fit_transform(cluster_input)
print(f'Divergence:', tsne_model.kl_divergence_)

# Results df
tsne_df = pd.DataFrame(y, columns=TSNE_N_COMPONENT_NAMES)

# Visualize
def tsne_chart(colors: list = None):

    fig = go.Figure()
    if TSNE_N_COMPONENTS == 2:
        fig = px.scatter(
            x=tsne_df['TSNE Component 1'],
            y=tsne_df['TSNE Component 2'],
            color=colors,
        )
    else:
        fig = px.scatter_3d(
            x=tsne_df['TSNE Component 1'],
            y=tsne_df['TSNE Component 2'],
            z=tsne_df['TSNE Component 3'],
            color=colors,
        )
    fig.update_layout(
        xaxis_title='TSNE Component 1', 
        yaxis_title='TSNE Component 2'
    )
        
    return fig

# Show
print(tsne_df.shape)
print(tsne_df.head().to_string())

fig = tsne_chart()
fig.show()

# Clustering

In [ ]:
''' Model '''

# Params
MODEL_NAME = 'Spectral Clustering'

PARAMS = {
    'Spectral Clustering': {
        'n_clusters': 6,
        'eigen_solver': "arpack",
        'affinity': "nearest_neighbors",
        'random_state': 42,
    }
}

model_params = PARAMS[MODEL_NAME]
print(model_params)

# Model
cluster_model = None
if MODEL_NAME == 'Spectral Clustering':
    cluster_model = SpectralClustering(**model_params)

# Fit
cluster_model.fit(cluster_input)

# Labels
labels = cluster_model.labels_
n_clusters_ = len(set(labels)) - (1 if -1 in labels else 0)
n_noise_ = list(labels).count(-1)

offense_tendencies['Cluster'] = labels

# Score
silhouette_score_ = silhouette_score(cluster_input, labels)

# Visualize
fig = tsne_chart(labels.astype(str))

params_str = ' | '.join([f'{key} = {val}' for key,val in model_params.items()])
fig.update_layout(
    title=f'{MODEL_NAME}<br><sup>{silhouette_score_ = :.4f} | {params_str}</sup>'
)
fig.show()

In [ ]:
''' Visualize Clusters '''

CLUSTER_COL = 'Cluster'
CLUSTERS_LIST = offense_tendencies[CLUSTER_COL].sort_values(ascending=True).unique().tolist()

# Average the features for each cluster
agg_dict: dict = {feature: ['min', 'mean', 'max'] for feature in FEATURES}
agg_dict[CLUSTER_COL] = 'size'

avgs_by_cluster = offense_tendencies.groupby(CLUSTER_COL).aggregate(agg_dict)
avgs_by_cluster.columns = [' '.join(col) for col in avgs_by_cluster.columns]
avgs_by_cluster = avgs_by_cluster.rename(columns={f'{CLUSTER_COL} size': '# Teams'})
print(avgs_by_cluster.head().to_string())


def get_feature_pct_scores(features: list, feature_vals: list):
    # Feature value percentiles
    pct_scores = []
    for i in range(len(features)):
        feature = features[i]
        val = feature_vals[i]
        pct_score = percentileofscore(offense_tendencies[feature].tolist(), val, kind='weak') / 100
        
        pct_scores.append(pct_score)
    
    return pct_scores


def visualize_cluster_features(cluster: int):

    features = FEATURES
    feature_cols = [f'{feature} mean' for feature in features]

    ## Data ##

    # Filter to cluster
    cluster_slice = offense_tendencies.loc[offense_tendencies[CLUSTER_COL] == cluster, :]
    cluster_avgs_slice = avgs_by_cluster.loc[avgs_by_cluster.index.get_level_values(CLUSTER_COL) == cluster, :]

    # Cluster stats
    n_teams = len(cluster_slice)
    cluster_record = cluster_slice[['G', 'W', 'L', 'T']].sum()
    win_pct = cluster_slice['W'].sum() / cluster_slice['G'].sum()
    record_str = f"{cluster_slice['W'].sum()}-{cluster_slice['L'].sum()}-{cluster_slice['T'].sum()}"
    
    # Feature averages
    cluster_feature_vals = cluster_avgs_slice[feature_cols].values.tolist()[0]
    cluster_feature_vals_fmt = []
    for i in range(len(features)):
        val = cluster_feature_vals[i]
        cluster_feature_vals_fmt.append(f'{val:.1%}') if features[i][0] == '%' else cluster_feature_vals_fmt.append(f'{val:,.2f}')
    
    # Feature percentiles
    cluster_feature_pctiles = get_feature_pct_scores(features=features, feature_vals=cluster_feature_vals)
    cluster_feature_pctiles_fmt = [f'{s:.1%}' for s in cluster_feature_pctiles]
    color_scale_len = len(px.colors.diverging.PRGn) - 1
    
    ## Figure ##

    # Radar
    radar = px.line_polar(
        r=cluster_feature_pctiles,
        theta=features,
        line_close=True,
        color_discrete_sequence=['#44546a'],
        
    )

    # Table
    pctile_colors = [px.colors.diverging.PRGn[int(p * color_scale_len)] for p in cluster_feature_pctiles]
    text_colors = []
    for p in cluster_feature_pctiles:
        if p <= .15 or p >= 0.85: text_colors.append('white')
        else: text_colors.append('#323232')

    tbl = go.Table(
        columnwidth=[2,1,1],
        header=dict(
            fill_color='#CCCCCC',
            font=dict(weight='bold'),
            line=dict(color='#323232', width=1),
            values=['Feature', 'Avg Value', 'Percentile'],
        ),
        cells=dict(
            fill_color=['white', 'white', pctile_colors],
            font=dict(weight=['bold', 'normal', 'normal'], color=['#323232', '#323232', text_colors]),
            line=dict(color='#323232', width=1),
            values=[features, cluster_feature_vals_fmt, cluster_feature_pctiles_fmt]
        )
    )
    
    # Figure
    fig = make_subplots(
        rows=1, cols=2,
        column_widths=[4,3],
        horizontal_spacing=0.1,
        specs=[[{"type": "polar"}, {"type": "domain"}]]
    )
    
    for trace in radar.data:
        fig.add_trace(
            trace, row=1, col=1
        )
    fig.add_trace(
        tbl, row=1, col=2
    )


    # Formatting
    fig.update_traces(
        fill='toself',
        opacity=0.7,
        mode='lines+markers+text',
        marker_size=7,
        row=1, col=1
    )
    
    fig.update_polars(
        bgcolor='#e1e1e1',
        angularaxis=dict(
            linecolor='#CCCCCC',
            showgrid=True,
            gridcolor='#fafafa',
            showticklabels=True,
            ticks=""
        ),
        radialaxis=dict(
            gridcolor='#fafafa',
        )
    )

    fig.update_layout(
        title_text=f"<b>Cluster {cluster}</b><br><sup>{n_teams = :,} | record = {record_str} ({win_pct:.3f})</sup>",
        polar=dict(radialaxis_range=(0,1)),
        margin=dict(b=50, r=50, l=75, t=75),
        height=500,
        width=1200,
        showlegend=True,
    )

    # Show
    fig.show()

    pcts = cluster_slice.sort_values(by='PCT', ascending=False)[FEATURES].rank(pct=True)
    print(pcts.head(10).to_string())


for cluster in CLUSTERS_LIST:
    visualize_cluster_features(cluster=cluster)